# Plot Figures in Hakim et al. (2026)

## Import libraries

In [ ]:
import logging

import numpy as np
import optimistix as optx

from atmodeller import (
    debug_logger,
)

logger = debug_logger()
logger.setLevel(logging.INFO)
# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from jaxmod.constants import AVOGADRO

from scipy.interpolate import interp1d
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"labellines.*")
warnings.filterwarnings("ignore", category=UserWarning, module=r"matplotlib.*")

# Comment below after installing labellines if not installed in the environment (needed only once))
!pip install matplotlib-label-lines

from labellines import labelLine, labelLines
from typing import Tuple, List, Dict, Any
from molmass import Formula

## Elemental Abundances

In [ ]:
# Palme and O'Neill (2014) Treatise on Geochemistry - Table 3

SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999


# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07


# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53

Primitive composition of the Earth's mantle \
Table 3 - Palme and O'Neill (2014) Treatise on Geochemistry

| Component  | Mass % |
|------------|--------|
| MgO        | 36.77  |
| Al2O3      | 4.49   |
| SiO2       | 45.4   |
| CaO        | 3.65   |
| FeO(t)     | 8.1    |
| Total      | 98.41  |
| Mg#        | 0.890  |


Present-day solar composition \
Table 8 - Lodders et al. (2009) Springer book chapter 

| Element    | Mass % |
|------------|--------|
| H (=X)     | 73.9   |
| He (=Y)    | 24.69  |
| O          | 0.63   |
| C          | 0.22   |
| Ne         | 0.17   |
| Fe         | 0.12   |
| N          | 0.07   |
| Si         | 0.07   |
| Mg         | 0.06   |
| S          | 0.03   |
| others     | 0.04   |
| total (=Z) | 1.41   |

Table 6 - Lodders et al. (2009) Springer book chapter 

| Element | A (log N(H) = 12) |
|---------|-------------------|
| H       | 12                |
| He      | 10.93             |
| C       | 8.39              |
| N       | 7.86              |
| O       | 8.73              |
| Ne      | 8.05              |
| Na      | 6.29              |
| Mg      | 7.54              |
| Al      | 6.46              |
| Si      | 7.53              |
| P       | 5.45              |
| S       | 7.16              |
| Cl      | 5.25              |
| Ar      | 6.5               |
| K       | 5.11              |
| Ca      | 6.31              |
| Ti      | 4.93              |
| V       | 3.99              |
| Fe      | 7.46              |


# Model Setup

## Planet Parameters

In [ ]:
# Mass and radius of TOI-421b
planet_mass = 6.7 * 5.972e24  # kg
MEB_radius = 1.65 * 6371000  # metre
atm_radius = 2.64 * 6371000  # metre
# MEB radius = 1.65 Earth radii in metre for 6.7 Earth masses (for atm_radius = 2.64 Earth radii)
# M-R relation from Hakim et al. (2018) Icarus

# Temperature of TOI-421b
MEB_temperature = 3000  # K
atm_temperature = 450  # K

## Elemental Budgets

In [ ]:
number_of_realisations = 50

# For envelope composition setup
hmps = np.logspace(-1, 0, num=number_of_realisations)  # wt% H
h_kgs = hmps / 100 * planet_mass  # kg

si_kg_magma: float = Si_massfraction_palme14 * planet_mass
o_kg_magma: float = O_massfraction_palme14 * planet_mass

# Lodders et al. (2009) Springer book chapter Table 8
si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09

# For atmosphere composition setup
atm_scalings = np.logspace(-10, 0, num=number_of_realisations)

h_kg = 1 / 100 * planet_mass  # kg for 1 wt% H

# Lodders et al. (2009) Springer book chapter Table 8
si_kg_solar = h_kg * Si_masspercent_lodders09 / H_masspercent_lodders09
o_kg_solar = h_kg * O_masspercent_lodders09 / H_masspercent_lodders09
c_kg_solar = h_kg * C_masspercent_lodders09 / H_masspercent_lodders09
n_kg_solar = h_kg * N_masspercent_lodders09 / H_masspercent_lodders09
he_kg_solar = h_kg * He_masspercent_lodders09 / H_masspercent_lodders09

# Plots published with Hakim et al. (2025)

## Setup plot colors

In [ ]:
color_tot = "black"
color_H2 = "orange"
color_H2O = "blue"
color_SiH4 = "red"
color_SiO = "brown"
color_O2 = "limegreen"
color_CO2 = "cyan"
color_CO = "magenta"
color_CH4 = "purple"
color_N2 = "green"
color_NH3 = "pink"
color_CHN = "olive"
color_He = "gray"
color_H = "orange"
color_C = "cyan"
color_N = "green"
color_O = "blue"
color_Si = "red"

## A. H-O-Si Envelope Composition

### Retrieve Data

In [ ]:
def load_magma_sol(init_metallicity: int, regime: str) -> Dict[str, Any]:
    """
    Load magma + solubility data into a structured dictionary.

    Parameters
    ----------
    init_metallicity : int
        e.g., 1 for '1xsolar'
    regime : str
        'ideal' or 'real'

    Returns
    -------
    dict
        Nested dict with keys: 'pressure', 'fugacity_coefficient', 'moles', 'mass'
    """
    filename = f"HOSi_magma_sol_{regime}_{init_metallicity}xsolar.xlsx"

    # Map sheet names to canonical keys in the output
    gas_sheets = {
        "H2_g":   "H2",
        "H2O_g":  "H2O",
        "O2_g":   "O2",
        "H4Si_g": "SiH4",
        "OSi_g":  "SiO",
        "state": "atmosphere",
    }

    element_sheets = {
        "element_H":  "H",
        "element_O":  "O",
        "element_Si": "Si",
    }

    # Read all needed sheets in one call; returns a dict of DataFrames
    xls = pd.read_excel(
        filename,
        sheet_name=list(gas_sheets.keys()) + list(element_sheets.keys()),
        engine="openpyxl"
    )

    data = {
        "pressure": {
            gas: xls[sheet]["pressure"]
            for sheet, gas in gas_sheets.items()
            if "pressure" in xls[sheet].columns
        },
        "fugacity_coefficient": {
            gas: xls[sheet]["fugacity_coefficient"]
            for sheet, gas in gas_sheets.items()
            if "fugacity_coefficient" in xls[sheet].columns
        },
        "moles": {
            "atmosphere": {
                el: xls[sheet]["gas_number"]
                for sheet, el in element_sheets.items()
            },
            "dissolved": {
                el: xls[sheet]["dissolved_number"]
                for sheet, el in element_sheets.items()
            },
            "total": {
                el: xls[sheet]["total_number"]
                for sheet, el in element_sheets.items()
            },
        },
        "mass": {
            "atmosphere": {
                el: xls[sheet]["gas_mass"]
                for sheet, el in element_sheets.items()
            },
            "dissolved": {
                el: xls[sheet]["dissolved_mass"]
                for sheet, el in element_sheets.items()
            },
            "total": {
                el: xls[sheet]["total_mass"]
                for sheet, el in element_sheets.items()
            },
        },
    }

    return data


init_metallicity = 1

ideal = load_magma_sol(init_metallicity, "ideal")
real  = load_magma_sol(init_metallicity, "real")

### Fig. 3 - Plot MEB Partial Pressures

In [ ]:
fig, (ax0, ax1) = plt.subplots(2, figsize=(6, 6), tight_layout="True")


ax0.plot(hmps, real["pressure"]["atmosphere"], color=color_tot, lw=4, ls="-", label="Total")
ax0.plot(hmps, real["pressure"]["H2"], color=color_H2, lw=3, ls="-", label="H$_2$")
ax0.plot(hmps, real["pressure"]["H2O"], color=color_H2O, lw=3, ls="-", label="H$_2$O")
ax0.plot(hmps, real["pressure"]["SiH4"], color=color_SiH4, lw=3, ls="-", label="SiH$_4$")
ax0.plot(hmps, real["pressure"]["SiO"], color=color_SiO, lw=3, ls="-", label="SiO")

ax0.set_title(r"(a) Real gases")
ax0.set_ylim([1e0, 2e5])
ax0.set_xscale("log")
ax0.set_yscale("log")
ax0.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax0.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax0.set_xticks([0.1, 0.3, 1])
ax0.get_xaxis().set_major_formatter(ScalarFormatter())

ax00 = ax0.twinx()
ax00.set_yscale("log")
ax00.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
ax00.set_ylim([1e-4, 2e1])

ax1.plot(hmps, ideal["pressure"]["atmosphere"], color=color_tot, lw=4, ls="-.", label="Total")
ax1.plot(hmps, ideal["pressure"]["H2"], color=color_H2, lw=3, ls="-.", label="H$_2$")
ax1.plot(hmps, ideal["pressure"]["H2O"], color=color_H2O, lw=3, ls="-.", label="H$_2$O")
ax1.plot(hmps, ideal["pressure"]["SiH4"], color=color_SiH4, lw=3, ls="-.", label="SiH$_4$")
ax1.plot(hmps, ideal["pressure"]["SiO"], color=color_SiO, lw=3, ls="-.", label="SiO")

ax1.set_title(r"(b) Ideal gases")
ax1.set_ylim([1e0, 2e5])
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax1.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax1.set_xticks([0.1, 0.3, 1])
ax1.get_xaxis().set_major_formatter(ScalarFormatter())

ax11 = ax1.twinx()
ax11.set_yscale("log")
ax11.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
ax11.set_ylim([1e-4, 2e1])

labelLines(ax0.get_lines())
labelLines(ax1.get_lines())

fig.suptitle(f"Envelope Composition at MEB (H$-$O$-$Si system)", fontsize=16)

plt.savefig(f"HOSi_envelope.pdf", bbox_inches="tight")
plt.savefig(f"HOSi_envelope.png", bbox_inches="tight")
plt.show()

### Fig. 4 - Plot Fugacity Coefficients

In [ ]:
fig, ax1 = plt.subplots(1, figsize=(6, 3), tight_layout="True")

(l1,) = ax1.plot(hmps, real["fugacity_coefficient"]["H2"], color=color_H2, lw=3, ls="-", label="H$_2$")
(l2,) = ax1.plot(hmps, real["fugacity_coefficient"]["H2O"], color=color_H2O, lw=3, ls="-", label="H$_2$O")
(l3,) = ax1.plot(hmps, real["fugacity_coefficient"]["SiH4"], color=color_SiH4, lw=3, ls="-", label="SiH$_4$")
(l4,) = ax1.plot(hmps, real["fugacity_coefficient"]["SiO"], color=color_SiO, lw=3, ls="-", label="SiO")
(l5,) = ax1.plot(hmps, real["fugacity_coefficient"]["O2"], color=color_O2, lw=3, ls="-", label="O$_2$")

ax1.set_title(f"Real Gas Fugacity Coefficients (H$-$O$-$Si system)")

ax1.set_ylim([8e-2, 2e2])
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax1.set_ylabel(r"Fugacity Coefficient", fontsize=12)
ax1.set_xticks([0.1, 0.3, 1])
ax1.get_xaxis().set_major_formatter(ScalarFormatter())

labelLines(plt.gca().get_lines())

plt.savefig(f"HOSi_fugacitycoeff.pdf", bbox_inches="tight")
plt.savefig(f"HOSi_fugacitycoeff.png", bbox_inches="tight")
plt.show()

### Fig. 5 - Plot H Partitioning in Magma

In [ ]:
fig, ax1 = plt.subplots(1, figsize=(6, 3), tight_layout="True")

ax1.plot(
    hmps,
    100 * real["mass"]["dissolved"]["H"] / real["mass"]["total"]["H"],
    color=color_tot,
    lw=3,
    ls="-",
    label="Real",
)
ax1.plot(
    hmps,
    100 * ideal["mass"]["dissolved"]["H"] / ideal["mass"]["total"]["H"],
    color=color_tot,
    lw=3,
    ls="-.",
    label="Ideal",
)

ax1.set_title(f"Hydrogen Solubility in Magma (H$-$O$-$Si system)")

ax1.set_ylim([-5, 105])
ax1.set_xscale("log")
ax1.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax1.set_ylabel(r"Partitioning in Magma [%]", fontsize=12)
ax1.set_xticks([0.1, 0.3, 1])
ax1.get_xaxis().set_major_formatter(ScalarFormatter())

labelLines(ax1.get_lines())

plt.savefig(f"HOSi_Hmagma.pdf", bbox_inches="tight")
plt.savefig(f"HOSi_Hmagma.png", bbox_inches="tight")
plt.show()

## B. H-He-C-N-O-Si Envelope composition

### Fig. 6 -  Ideal vs Real - Envelope Composition, Fugacity Coefficients, Volatile Partitioning in Magma

In [ ]:
mantle_melt_fraction = 1
init_metallicity = 1

files = {
    "ideal": f"HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}.xlsx",
    "real": f"HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}.xlsx"
}

# Gas species and their sheet names
gas_species = ["CO_g", "H2_g", "H2O_g", "O2_g", "H4Si_g", "OSi_g", "He_g", "CO2_g", "CH4_g", "N2_g", "H3N_g"]

elements = ["H", "He", "C", "N"]

# Dictionary to hold data
data = {"ideal": {}, "real": {}}

for model, filename in files.items():
    # Read pressures and fugacity coefficients for gases
    data[model]["pressure"] = {gas: pd.read_excel(filename, sheet_name=gas)["pressure"] for gas in gas_species}
    data[model]["fugacity_coefficient"] = {gas: pd.read_excel(filename, sheet_name=gas)["fugacity_coefficient"] for gas in gas_species}

    data[model]["total_pressure"] = pd.read_excel(filename, sheet_name="state")["pressure"]

    # Element masses
    data[model]["element_atm_mass"] = {el: pd.read_excel(filename, sheet_name=f"element_{el}")["gas_mass"] for el in elements}
    data[model]["element_total_mass"] = {el: pd.read_excel(filename, sheet_name=f"element_{el}")["total_mass"] for el in elements}

# Colors for species
colors = {
    "CO_g": color_CO, "H2_g": color_H2, "H2O_g": color_H2O, "O2_g": color_O2, "H4Si_g": color_SiH4,
    "OSi_g": color_SiO, "He_g": color_He, "CO2_g": color_CO2, 
    "CH4_g": color_CH4, "N2_g": color_N2, "H3N_g": color_NH3
}

labels = {
    "CO_g": "CO", "H2_g": "H$_2$", "H2O_g": "H$_2$O", "O2_g": "O$_2$", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True)

# (a) Ideal Gas Pressures
for gas in colors:
    ax0.plot(hmps, data["ideal"]["pressure"][gas], color=colors[gas], lw=2, ls="-.", label=labels[gas])
ax0.plot(hmps, data["ideal"]["total_pressure"], color=color_tot, lw=3, ls="-.", label="Total")
ax0.set_title(r"(a) Envelope Composition at MEB (Ideal)")
ax0.set_xscale("log"); ax0.set_yscale("log")
ax0.set_ylim([1e0, 2e5])
ax0.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax0.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax0.set_xticks([0.1, 0.3, 1]); ax0.get_xaxis().set_major_formatter(ScalarFormatter())

# (b) Real Gas Pressures
for gas in colors:
    ax1.plot(hmps, data["real"]["pressure"][gas], color=colors[gas], lw=2, ls="-", label=labels[gas])
ax1.plot(hmps, data["real"]["total_pressure"], color=color_tot, lw=3, ls="-", label="Total")
ax1.set_title(r"(b) Envelope Composition at MEB (Real)")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_ylim([1e0, 2e5])
ax1.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax1.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax1.set_xticks([0.1, 0.3, 1]); ax1.get_xaxis().set_major_formatter(ScalarFormatter())

# Twin axes for GPa
for ax in [ax0, ax1]:
    twin = ax.twinx()
    twin.set_yscale("log")
    twin.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
    twin.set_ylim([1e-4, 2e1])

# (c) Volatile Solubility
elements = {"H": color_H, "He": color_He, "C": color_C, "N": color_N}
for el, col in elements.items():
    ax2.plot(hmps, 100*(data["real"]["element_total_mass"][el]-data["real"]["element_atm_mass"][el])/data["real"]["element_total_mass"][el],
             color=col, lw=2, ls="-", label=el)
    ax2.plot(hmps, 100*(data["ideal"]["element_total_mass"][el]-data["ideal"]["element_atm_mass"][el])/data["ideal"]["element_total_mass"][el],
             color=col, lw=2, ls="-.")
ax2.set_title(r"(c) Volatile Solubility in Magma (Ideal vs Real)")
ax2.set_xscale("log"); ax2.set_ylim([-5, 105])
ax2.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax2.set_ylabel(r"Partitioning in Magma [%]", fontsize=12)
ax2.set_xticks([0.1, 0.3, 1]); ax2.get_xaxis().set_major_formatter(ScalarFormatter())

# (d) Fugacity Coefficients
for gas in colors:
    ax3.plot(hmps, data["real"]["fugacity_coefficient"][gas], color=colors[gas], lw=2, ls="-", label=labels[gas])
ax3.set_title(r"(d) Real Gas Fugacity Coefficients")
ax3.set_xscale("log"); ax3.set_yscale("log")
ax3.set_ylim([8e-2, 1e5])
ax3.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax3.set_ylabel(r"Fugacity Coefficient", fontsize=12)
ax3.set_xticks([0.1, 0.3, 1]); ax3.get_xaxis().set_major_formatter(ScalarFormatter())

# label lines
labelLines(ax0.get_lines())
labelLines(ax1.get_lines())
labelLines(ax2.get_lines())
labelLines(ax3.get_lines())

custom_handles = [
    Line2D([0], [0], color="black", ls="-.", lw=2, label="Ideal"),
    Line2D([0], [0], color="black", ls="-", lw=2, label="Real"),
]

ax2.legend(
    handles=custom_handles,
    bbox_to_anchor=(0.01, 0.41),
    loc="upper left",
    ncol=1,
    labelspacing=0.1,
    borderpad=0.2,
    columnspacing=0.1,
    framealpha=0.5,
)

fig.suptitle(r"Envelope Composition at MEB for Ideal vs Real Gases (H$-$He$-$C$-$N$-$O$-$Si system, 1$\times$ solar)", fontsize=16)

plt.savefig("HHeCNOSi_ideal_real.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_ideal_real.png", bbox_inches="tight")
plt.show()

### Fig. A1 - Full Range - Ideal vs Real - Envelope Composition

In [ ]:
mantle_melt_fraction = 1
init_metallicity = 1

files = {
    "ideal": f"HHeCNOSi_magma_sol_ideal_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}.xlsx",
    "real": f"HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}.xlsx"
}

# Gas species and their sheet names
gas_species = ["H2_g", "H2O_g", "O2_g", "H4Si_g", "OSi_g", "He_g", "CO2_g", "CO_g", "CH4_g", "N2_g", "H3N_g"]

# Dictionary to hold data
data = {"ideal": {}, "real": {}}

for model, filename in files.items():
    # Read pressures and fugacity coefficients for gases
    data[model]["pressure"] = {gas: pd.read_excel(filename, sheet_name=gas)["pressure"] for gas in gas_species}
    data[model]["fugacity_coefficient"] = {gas: pd.read_excel(filename, sheet_name=gas)["fugacity_coefficient"] for gas in gas_species}

    data[model]["total_pressure"] = pd.read_excel(filename, sheet_name="state")["pressure"]

    # Element masses
    data[model]["element_atm_mass"] = {el: pd.read_excel(filename, sheet_name=f"element_{el}")["gas_mass"] for el in elements}
    data[model]["element_total_mass"] = {el: pd.read_excel(filename, sheet_name=f"element_{el}")["total_mass"] for el in elements}

# Colors for species
colors = {
    "H2_g": color_H2, "H2O_g": color_H2O, "O2_g": color_O2, "H4Si_g": color_SiH4,
    "OSi_g": color_SiO, "He_g": color_He, "CO2_g": color_CO2, "CO_g": color_CO, 
    "CH4_g": color_CH4, "N2_g": color_N2, "H3N_g": color_NH3
}

labels = {
    "H2_g": "H$_2$", "H2O_g": "H$_2$O", "O2_g": "O$_2$", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", "CO_g": "CO", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}

fig, ((ax0, ax1)) = plt.subplots(1, 2, figsize=(10, 3), tight_layout=True)

# (a) Ideal Gas Pressures
ax0.plot(hmps, data["ideal"]["total_pressure"], color=color_tot, lw=3, ls="-.", label="Total")
for gas in colors:
    ax0.plot(hmps, data["ideal"]["pressure"][gas], color=colors[gas], lw=2, ls="-.", label=labels[gas])
ax0.set_title(r"(a) Envelope Composition at MEB (Ideal)")
ax0.set_xscale("log"); ax0.set_yscale("log")
ax0.set_ylim([2e-11, 2e5])
ax0.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax0.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax0.set_xticks([0.1, 0.3, 1]); ax0.get_xaxis().set_major_formatter(ScalarFormatter())

# (b) Real Gas Pressures
ax1.plot(hmps, data["real"]["total_pressure"], color=color_tot, lw=3, ls="-", label="Total")
for gas in colors:
    ax1.plot(hmps, data["real"]["pressure"][gas], color=colors[gas], lw=2, ls="-", label=labels[gas])
ax1.set_title(r"(b) Envelope Composition at MEB (Real)")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_ylim([2e-11, 2e5])
ax1.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
ax1.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
ax1.set_xticks([0.1, 0.3, 1]); ax1.get_xaxis().set_major_formatter(ScalarFormatter())


# Twin axes for GPa
for ax in [ax0, ax1]:
    twin = ax.twinx()
    twin.set_yscale("log")
    twin.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
    twin.set_ylim([2e-15, 2e1])

labelLines(ax0.get_lines())
labelLines(ax1.get_lines())

fig.suptitle(
    r"Envelope Composition at MEB for Ideal vs Real Gases (H$-$He$-$C$-$N$-$O$-$Si system, 1$\times$ solar)",
    fontsize=16,
)

plt.savefig("HHeCNOSi_ideal_real_fullrange.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_ideal_real_fullrange.png", bbox_inches="tight")
plt.show()

### Fig. 7 - Effects of Mantle Melt Fraction and Metallicity - Envelope Composition

In [ ]:
mantle_melt_fractions = [1, 0.01]  # 100% and 1%
metallicities = [1, 100]  # 1x and 100x solar

# Gas species and their sheet names
gas_species = ["CO_g", "H2_g", "H2O_g", "O2_g", "H4Si_g", "OSi_g", "He_g", "CO2_g", "CH4_g", 
               "N2_g", "H3N_g"]

# load pressures
def load_pressures(metallicity, melt_fraction):
    filename = f"HHeCNOSi_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}.xlsx"
    data = {}
    data["total_pressure"] = pd.read_excel(filename, sheet_name="state")["pressure"]
    for gas in gas_species:
        data[gas] = pd.read_excel(filename, sheet_name=gas)["pressure"]
    return data


# Colors for species
colors = {
    "H2_g": color_H2, "H2O_g": color_H2O, "O2_g": color_O2, "H4Si_g": color_SiH4,
    "OSi_g": color_SiO, "He_g": color_He, "CO2_g": color_CO2, "CO_g": color_CO, 
    "CH4_g": color_CH4, "N2_g": color_N2, "H3N_g": color_NH3
}

labels = {
    "H2_g": "H$_2$", "H2O_g": "H$_2$O", "O2_g": "O$_2$", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", "CO_g": "CO", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}


fig, axes = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True)

# Loop through melt fractions and metallicities
for row, mf in enumerate(mantle_melt_fractions):
    for col, met in enumerate(metallicities):
        ax = axes[row, col]
        data = load_pressures(met, mf)

        for i, gas in enumerate(gas_species):
            ax.plot(hmps, data[gas], c=colors[gas], ls='-', lw=2, label=labels[gas])
        ax.plot(hmps, data["total_pressure"], color=color_tot, lw=3, ls="-", label="Total")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
        ax.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
        ax.set_xticks([0.1, 0.3, 1])
        ax.get_xaxis().set_major_formatter(ScalarFormatter())
        ax.set_ylim([1e0, 2e5])
        labelLines(ax.get_lines())
        twin = ax.twinx()
        twin.set_yscale("log")
        twin.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
        twin.set_ylim([1e-4, 2e1])

axes[0,0].set_title(r"(a) 1$\times$ solar, 100% melt")
axes[0,1].set_title(r"(b) 100$\times$ solar, 100% melt")
axes[1,0].set_title(r"(c) 1$\times$ solar, 1% melt")
axes[1,1].set_title(r"(d) 100$\times$ solar, 1% melt")

fig.suptitle(r"Envelope Composition at MEB (H$-$He$-$C$-$N$-$O$-$Si system)", fontsize=16)

plt.savefig("HHeCNOSi_envelope.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_envelope.png", bbox_inches="tight")
plt.show()

### Fig. A2 - Full Range - Effects of Mantle Melt Fraction and Metallicity - Envelope Composition

In [ ]:
mantle_melt_fractions = [1, 0.01]  # 100% and 1%
metallicities = [1, 100]  # 1x and 100x solar

# Gas species and their sheet names
gas_species = ["H2_g", "H2O_g", "O2_g", "H4Si_g", "OSi_g", "He_g", "CO2_g", "CO_g", "CH4_g", 
               "N2_g", "H3N_g"]

# load pressures
def load_pressures(metallicity, melt_fraction):
    filename = f"HHeCNOSi_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}.xlsx"
    data = {}
    data["total_pressure"] = pd.read_excel(filename, sheet_name="state")["pressure"]
    for gas in gas_species:
        data[gas] = pd.read_excel(filename, sheet_name=gas)["pressure"]
    return data


# Colors for species
colors = {
    "H2_g": color_H2, "H2O_g": color_H2O, "O2_g": color_O2, "H4Si_g": color_SiH4,
    "OSi_g": color_SiO, "He_g": color_He, "CO2_g": color_CO2, "CO_g": color_CO, 
    "CH4_g": color_CH4, "N2_g": color_N2, "H3N_g": color_NH3
}

labels = {
    "H2_g": "H$_2$", "H2O_g": "H$_2$O", "O2_g": "O$_2$", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", "CO_g": "CO", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}


fig, axes = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True)

# Loop through melt fractions and metallicities
for row, mf in enumerate(mantle_melt_fractions):
    for col, met in enumerate(metallicities):
        ax = axes[row, col]
        data = load_pressures(met, mf)

        ax.plot(hmps, data["total_pressure"], color=color_tot, lw=3, ls="-", label="Total")
        for i, gas in enumerate(gas_species):
            ax.plot(hmps, data[gas], c=colors[gas], ls='-', lw=2, label=labels[gas])
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
        ax.set_ylabel(r"MEB Partial Pressure [bar]", fontsize=12)
        ax.set_xticks([0.1, 0.3, 1])
        ax.get_xaxis().set_major_formatter(ScalarFormatter())
        ax.set_ylim([2e-11, 2e5])
        labelLines(ax.get_lines())
        twin = ax.twinx()
        twin.set_yscale("log")
        twin.set_ylabel(r"MEB Partial Pressure [GPa]", fontsize=12)
        twin.set_ylim([2e-15, 2e1])

axes[0,0].set_title(r"(a) 1$\times$ solar, 100% melt")
axes[0,1].set_title(r"(b) 100$\times$ solar, 100% melt")
axes[1,0].set_title(r"(c) 1$\times$ solar, 1% melt")
axes[1,1].set_title(r"(d) 100$\times$ solar, 1% melt")

fig.suptitle(r"Envelope Composition at MEB (H$-$He$-$C$-$N$-$O$-$Si system)", fontsize=16)

plt.savefig("HHeCNOSi_envelope_fullrange.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_envelope_fullrange.png", bbox_inches="tight")
plt.show()

### Fig. B1 - Effects of Mantle Melt Fraction and Metallicity - Volatile Partitioning in Magma

In [ ]:
mantle_melt_fractions = [1, 0.01]  # 100% and 1%
metallicities = [1, 100]  # 1x and 100x solar

# Elements
elements = ["H", "He", "C", "N"]


# load pressures
def load_solubilities(metallicity, melt_fraction):
    filename = f"HHeCNOSi_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}.xlsx"
    data = {}
    for el in elements:
        data[el] = 100 * (
            1 - pd.read_excel(filename, sheet_name=f"element_{el}")["gas_mass"] 
            / pd.read_excel(filename, sheet_name=f"element_{el}")["total_mass"]
            )
    return data


# Colors for species
colors = {
    "H": color_H, "He": color_He, "C": color_C, "N": color_N
}

labels = {
    "H": "H", "He": "He", "C": "C", "N": "N"
}


fig, axes = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True)

# Loop through melt fractions and metallicities
for row, mf in enumerate(mantle_melt_fractions):
    for col, met in enumerate(metallicities):
        ax = axes[row, col]
        data = load_solubilities(met, mf)

        for i, el in enumerate(elements):
            ax.plot(hmps, data[el], c=colors[el], ls='-', lw=2, label=labels[el])
        ax.set_xscale("log")
        ax.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
        ax.set_ylabel(r"Partitioning in Magma [%]", fontsize=12)
        ax.set_xticks([0.1, 0.3, 1])
        ax.get_xaxis().set_major_formatter(ScalarFormatter())
        ax.set_ylim([-5, 105])
        labelLines(ax.get_lines())

axes[0,0].set_title(r"(a) 1$\times$ solar, 100% melt")
axes[0,1].set_title(r"(b) 100$\times$ solar, 100% melt")
axes[1,0].set_title(r"(c) 1$\times$ solar, 1% melt")
axes[1,1].set_title(r"(d) 100$\times$ solar, 1% melt")

fig.suptitle(r"Volatile Solubility in Magma (H$-$He$-$C$-$N$-$O$-$Si system)", fontsize=16)

plt.savefig("HHeCNOSi_solubility.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_solubility.png", bbox_inches="tight")
plt.show()

### Fig. B2 - Effects of Mantle Melt Fraction and Metallicity - Fugacity Coefficients

In [ ]:
mantle_melt_fractions = [1, 0.01]  # 100% and 1%
metallicities = [1, 100]  # 1x and 100x solar

# Gas species and their sheet names
gas_species = ["H2_g", "H2O_g", "O2_g", "H4Si_g", "OSi_g", "He_g", "CO2_g", "CO_g", "CH4_g", 
               "N2_g", "H3N_g"]

# load fugacity coefficients
def load_fugacity_coefficients(metallicity, melt_fraction):
    filename = f"HHeCNOSi_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}.xlsx"
    data = {}
    for gas in gas_species:
        data[gas] = pd.read_excel(filename, sheet_name=gas)["fugacity_coefficient"]
    return data


# Colors for species
colors = {
    "H2_g": color_H2, "H2O_g": color_H2O, "O2_g": color_O2, "H4Si_g": color_SiH4,
    "OSi_g": color_SiO, "He_g": color_He, "CO2_g": color_CO2, "CO_g": color_CO, 
    "CH4_g": color_CH4, "N2_g": color_N2, "H3N_g": color_NH3
}

labels = {
    "H2_g": "H$_2$", "H2O_g": "H$_2$O", "O2_g": "O$_2$", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", "CO_g": "CO", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}


fig, axes = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True)

# Loop through melt fractions and metallicities
for row, mf in enumerate(mantle_melt_fractions):
    for col, met in enumerate(metallicities):
        ax = axes[row, col]
        data = load_fugacity_coefficients(met, mf)

        for i, gas in enumerate(gas_species):
            ax.plot(hmps, data[gas], c=colors[gas], ls='-', lw=2, label=labels[gas])
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(r"Hydrogen Mass / Planet Mass [%]", fontsize=14)
        ax.set_ylabel(r"Fugacity Coefficient", fontsize=12)
        ax.set_xticks([0.1, 0.3, 1])
        ax.get_xaxis().set_major_formatter(ScalarFormatter())
        ax.set_ylim([8e-2, 1e6])
        labelLines(ax.get_lines())

axes[0,0].set_title(r"(a) 1$\times$ solar, 100% melt")
axes[0,1].set_title(r"(b) 100$\times$ solar, 100% melt")
axes[1,0].set_title(r"(c) 1$\times$ solar, 1% melt")
axes[1,1].set_title(r"(d) 100$\times$ solar, 1% melt")

fig.suptitle(r"Real Gas Fugacity Coefficients (H$-$He$-$C$-$N$-$O$-$Si system)", fontsize=16)

plt.savefig("HHeCNOSi_fugcoeff.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_fugcoeff.png", bbox_inches="tight")
plt.show()

### Fig. 9 - Envelope Ratios in units of Metallicity

In [ ]:
# Parameters
metallicities = [1, 10, 100]
melt_percents = [100, 30, 10, 3, 1]  # 100%, 30%, 10%, 3%, 1%
elements = ["H", "He", "O", "C", "N", "Si"]
elements_to_plot = ["He", "O", "C", "N", "Si"]

# Dictionary with elemental abundances (assumed defined elsewhere)
logN = {"H": H_logN, "He": He_logN, "O": O_logN, "C": C_logN, "N": N_logN, "Si": Si_logN}

# Read data into nested dictionary: data[metallicity][melt_fraction][element]
data = {}
for m in metallicities:
    data[m] = {}
    for mp in melt_percents:
        filename = f"HHeCNOSi_magma_sol_real_{m}xsolar_melt{mp}.xlsx"
        data[m][mp] = {
            el: pd.read_excel(filename, sheet_name=f"element_{el}")["gas_number"]
            for el in elements
        }

# Compute ratios for all metallicities and melt fractions
ratios_all = {}
for m in metallicities:
    ratios_all[m] = {}
    for mp in melt_percents:
        ratios = {}
        H_val = data[m][mp]["H"][number_of_realisations-1]  # Assuming index 49 for now
        for el in elements:
            if el != "H":
                ratios[el] = data[m][mp][el][number_of_realisations-1] / H_val / (10 ** (logN[el] - logN["H"]))
        ratios_all[m][mp] = ratios


# Colors and markers
element_colors = {
    "He": color_He,
    "C": color_C,
    "N": color_N,
    "O": color_O,
    "Si": color_Si,
}
markers = {1: "o", 10: "s", 100: "D"}

fig, ax1 = plt.subplots(1, figsize=(7, 4), tight_layout=True)

# Reference lines
ax1.axhline(y=1, color="black", ls="-")
ax1.axhline(y=10, color="black", ls="--")
ax1.axhline(y=100, color="black", ls=":")

plt.text(40, 1.5, r"1$\times$ solar")
plt.text(40, 15, r"10$\times$ solar")
plt.text(40, 150, r"100$\times$ solar")

# Scatter points dynamically
for mp in melt_percents:
    for m in metallicities:
        if mp not in ratios_all[m]:
            continue
        for el in element_colors.keys():
            y_val = ratios_all[m][mp][el]
            ax1.scatter(mp, y_val, color=element_colors[el], s=30, marker=markers[m])

# Axis settings
ax1.set_title(r"Envelope Elemental Ratios with Magma$-$Envelope Coupling")
ax1.set_xlim([9e-1, 1.1e2])
ax1.set_ylim([1e-4, 1e5])
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel(r"Mantle Melt [%]", fontsize=14)
ax1.set_ylabel(r"Elemental Ratios [$\times$ solar]", fontsize=14)

# Legend
custom_handles = [
    Line2D([], [], color="black", marker="D", ls="None", label=r"100$\times$ solar"),
    Line2D([], [], color="black", marker="s", ls="None", label=r"10$\times$ solar"),
    Line2D([], [], color="black", marker="o", ls="None", label=r"1$\times$ solar"),
] + [Line2D([0], [0], color=c, ls="-", lw=3, label=f"{el}/H") for el, c in element_colors.items()]


# Annotations
ax1.text(1.2, 0.03, "Solubility M > Solubility H\nFugacity > Partial Pressure")
ax1.annotate("", xy=(1.02, 2e-3), xytext=(1.02, 2e-1), arrowprops=dict(facecolor="black"))

ax1.text(1.2, 1e3, "Magma Vapourisation\nSolubility M < Solubility H\nFugacity < Partial Pressure")
ax1.annotate("", xy=(1.02, 1e5), xytext=(1.02, 1e3), arrowprops=dict(facecolor="black"))

ax1.legend(
    handles=custom_handles,
    handlelength=1,
    ncol=2,
    labelspacing=0.2,
    borderpad=0.2,
    columnspacing=0.5,
    bbox_to_anchor=(0.5, -0.01),
    loc="lower center",
)

plt.savefig("HHeCNOSi_ratios.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_ratios.png", bbox_inches="tight")
plt.show()

## C. H-He-C-N-O-Si Atmosphere Composition

### Plots atm speciation with condensation

In [ ]:
# Parameters
init_metallicities = [1, 100]  
mantle_melt_fractions = [1.0, 0.01, 0.0]  

# Gas species and corresponding sheet names
species_sheets = {
    "H2": "H2_g",
    "He": "He_g",
    "H2O": "H2O_g",
    "SiH4": "H4Si_g",
    "SiO": "OSi_g",
    "CO2": "CO2_g",
    "CO": "CO_g",
    "CH4": "CH4_g",
    "N2": "N2_g",
    "NH3": "H3N_g",
}

# Dictionary to store results
data = {}

for metallicity in init_metallicities:
    for melt_fraction in mantle_melt_fractions:

        # Build filename
        filename = f"HHeCNOSi_atm_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}_1wtH.xlsx"
        
        # Store pressure and molar mass
        key_prefix = f"{metallicity}xsolar_melt{round(melt_fraction * 100)}"
        data[f"pressure_{key_prefix}"] = pd.read_excel(filename, sheet_name="state")["pressure"]
        data[f"mu_{key_prefix}"] = 1e3 * pd.read_excel(filename, sheet_name="gas")["molar_mass"]
        
        # Read each species
        for species, sheet in species_sheets.items():
            df = pd.read_excel(filename, sheet_name=sheet)
            data[f"{species}_vmr_{key_prefix}"] = df["volume_mixing_ratio"]


# Colors for species
species_colors = {
    "H2": color_H2,
    "He": color_He,
    "H2O": color_H2O,
    "SiH4": color_SiH4,
    "SiO": color_SiO,
    "CH4": color_CH4,
    "CO": color_CO,
    "CO2": color_CO2,
    "N2": color_N2,
    "NH3": color_NH3
}

labels = {
    "CO_g": "CO", "H2_g": "H$_2$", "H2O_g": "H$_2$O", "H4Si_g": "SiH$_4$",
    "OSi_g": "SiO", "He_g": "He", "CO2_g": "CO$_2$", 
    "CH4_g": "CH$_4$", "N2_g": "N$_2$", "H3N_g": "NH$_3$"
}

species_list = list(species_sheets.keys())

# Create subplots 
fig, axes = plt.subplots(3, 2, figsize=(10, 9), tight_layout=True)

for row, melt_fraction in enumerate(mantle_melt_fractions):
    for col, metallicity in enumerate(init_metallicities):
        ax = axes[row, col]
        key_prefix = f"{metallicity}xsolar_melt{round(melt_fraction * 100)}"
        ax.axvspan(10**-4.95, 10**-2.64,  color=color_H2O, alpha=0.2,
                label="H$_2$O (Davenport et al. 2025)")
        ax.axvspan(10**-7.93, 10**-2.42, color=color_CO, alpha=0.2,
                label="CO (Davenport et al. 2025)")
        
        gases_obs, lss = ax.get_legend_handles_labels()

        # Plot each species
        for species in species_list:
            vmr_key = f"{species}_vmr_{key_prefix}"
            pressure_key = f"pressure_{key_prefix}"

            if vmr_key in data and pressure_key in data:
                ax.plot(
                    data[vmr_key], 
                    data[pressure_key],
                    color=species_colors[species],
                    lw=2,
                    ls="-",
                    label=labels[species_sheets[species]],
                )
    
        # Axis settings
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim([1e-12, 2e0])
        ax.set_ylim([1e2, 1e-3])
        ax.set_xlabel("Mixing Ratio", fontsize=14)
        ax.set_ylabel("Pressure [bar]", fontsize=14)



offsets = {}

for row, melt_fraction in enumerate(mantle_melt_fractions):
    for col, metallicity in enumerate(init_metallicities):
        ax = axes[row, col]
        lines = ax.get_lines()
        
        offsets[(row, col)] = []
        i=0
        for line in lines:
            i=i+1
            xdata, ydata = line.get_xdata(), line.get_ydata()
            
            # Choose mid-point for labeling
            mid_idx = len(xdata) // 2 + i
            x_mid, y_mid = xdata[mid_idx], ydata[mid_idx]
            
            # Store offsets 
            offsets[(row, col)].append((x_mid, y_mid))        

            labelLines(
                ax.get_lines(),
                align=False,
                xvals=[x for x, y in offsets[(row, col)]],
                yoffsets=[y * 0.1 for x, y in offsets[(row, col)]] 
            )

# Twin axis settings
mmw_span_free = (2.3, 2.7)
mmw_span_eq = (2.3, 6.3)

for row, melt_fraction in enumerate(mantle_melt_fractions):
    for col, metallicity in enumerate(init_metallicities):
        ax = axes[row, col]
        key_prefix = f"{metallicity}xsolar_melt{round(melt_fraction * 100)}"
        
        # Create twin axis
        twin_ax = ax.twiny()
        twin_ax.set_xlim([1.2, 20])
        twin_ax.set_xlabel(r"Mean Molecular Weight [amu]", fontsize=12)

        twin_ax.axvspan(*mmw_span_free, alpha=0.5, color="black", label="MMW (Free Chem)")
        twin_ax.axvspan(*mmw_span_eq, alpha=0.2, color="black", label="MMW (Eq. Chem)")

        mmw_obs, lss2 = twin_ax.get_legend_handles_labels()
        
        # Plot MMW
        mu_key = f"mu_{key_prefix}"
        pressure_key = f"pressure_{key_prefix}"
        
        twin_ax.plot(
            data[mu_key], 
            data[pressure_key],
            color="black",
            lw=3,
            ls="-",
            label="MMW",
            zorder=1
        )
        labelLines(twin_ax.get_lines(), align=True, yoffsets=0.1)



axes[0,0].set_title(f"(a) 1 wt% H, 1× solar, 100% melt")
axes[0,1].set_title(f"(b) 1 wt% H, 100× solar, 100% melt")
axes[1,0].set_title(f"(c) 1 wt% H, 1× solar, 1% melt")
axes[1,1].set_title(f"(d) 1 wt% H, 100× solar, 1% melt")
axes[2,0].set_title(f"(e) 1 wt% H, 1× solar, 0% melt")
axes[2,1].set_title(f"(f) 1 wt% H, 100× solar, 0% melt")


fig.legend(
    handles=mmw_obs + gases_obs,
    ncol=2,
    bbox_to_anchor=(0.5, -0.07),
    labelspacing=0,
    borderpad=0,
    columnspacing=0.1,
    fontsize=12,
    loc="lower center",
)
fig.suptitle(
    r"Equilibrium Atmospheric Composition with Condensation", fontsize=16
)

plt.savefig("HHeCNOSi_atmosphere.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_atmosphere.png", bbox_inches="tight")
plt.show()


### Plot condensate number densities and compare with FastChem

In [ ]:
def load_condensation(filename) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Reads a fastchem condensation file
    """

    cond = pd.read_csv(filename, sep=r"\s+")
    cond = cond.rename(
        columns={
            r"#p(bar)": "P(bar)",
        }
    )
    _condensates = list(cond.drop([r"T(K)", r"P(bar)"], axis=1).keys())
    condensates = []
    _elem_names = []

    for cond_name in _condensates:
        if (
            cond_name.endswith(r"(s,l)")
            or cond_name.endswith(r"(s)")
            or cond_name.endswith(r"(l)")
        ):
            condensates.append(cond_name)
        else:
            _elem_names.append(cond_name)
    elem_condfrac = cond[_elem_names]

    return cond, elem_condfrac, condensates

def formula_to_latex(formula: str) -> str:
    """
    Converts a chemical formula (e.g. SiO, O1Si1) into latex markdown.
    """
    # lazy bugfix
    if formula == "e-":
        return "e-"

    # extract formula; FastChem has suffixes to formulas which we want to avoid at this stage.
    suffix = None
    if "_" in formula:
        (formula, suffix) = formula.split("_")
    if formula.endswith("trans"):
        formula = formula[:-5]
        suffix = "trans"
    if formula.endswith("cis"):
        formula = formula[:-4]
        suffix = "cis"
    if "(s)" in formula:
        (formula, _) = formula.split("(s)")
        suffix = "s"
    if "(s,l)" in formula:
        (formula, _) = formula.split("(s,l)")
        suffix = "s,l"
    if "(l)" in formula:
        (formula, _) = formula.split("(l)")
        suffix = "l"

    # extract stoichiometry
    comp: Dict[str, Tuple[float, float, float]] = (
        Formula(formula).composition().asdict()
    )

    # most commonly, O is in the end of the formula
    oxy_string = ""
    if "O" in comp:
        oxy_string += "O"
        oxy_stoich: int = comp.pop("O")[0]
        if oxy_stoich > 1:
            oxy_string += "$_{" + str(oxy_stoich) + "}$"

    # comp is already sorted in Hill notation
    latex_formula: str = ""
    for elem, data in comp.items():
        stoich: int = data[0]
        if elem == "e-":
            if stoich > 0:
                latex_formula += "-"
            else:
                latex_formula += "+"
        else:
            latex_formula += elem
            if stoich > 1:
                latex_formula += "$_{" + str(stoich) + "}$"

    latex_formula += oxy_string

    if suffix is not None:
        latex_formula += f"({suffix})"

    return latex_formula

condensates_of_interest_fastchem = ["SiO2(s,l)", "C(s)", "SiC(s)", "Si(s,l)", "Si3N4(s,l)"]
condensates_of_interest_atmodeller = ["O2Si_cd", "C_cd", "CSi_cd", "Si_cd", "N4Si3_cd"]

species_colors = [color_SiO, color_C, color_H, color_Si, color_N, color_H2O]

cond_colors = dict(zip(condensates_of_interest_atmodeller, species_colors)) | dict(
    zip(condensates_of_interest_fastchem, species_colors)
)

fig, ax = plt.subplots(
    2, 2, figsize=(10, 6), tight_layout="True"
)

for i, melt_frac in enumerate(["100", "1"]):
    for j, metallicity in enumerate(["1", "100"]):

        # ------ Plot atmodeller condensates -------- #
        xls_file = pd.ExcelFile(
            f"HHeCNOSi_atm_magma_sol_real_{metallicity}xsolar_melt{melt_frac}_1wtH.xlsx"
        )
        xls_data = pd.read_excel(xls_file, sheet_name=None, index_col=0)
        pressures = xls_data["state"]["pressure"]
        volumes = xls_data["gas"]["volume"]
        mask = pressures <= 10000

        for condensate in condensates_of_interest_atmodeller:
            condensed_number = xls_data[f"{condensate}"]["total_number"]
            ax[i, j].plot(
                condensed_number[mask] / 1e6 / volumes[mask] * AVOGADRO,
                pressures[mask],
                label=formula_to_latex(condensate),
                color=cond_colors[condensate],
                lw=1,
            )

        # # ------- Plot fastchem species -------- #
        # outdir = f"fastchem_output/xmelt_{melt_frac}/{metallicity}xsolar/"
        # cond, elem_condfrac, condensate_names = load_condensation(
        #     outdir + "cond_chem.dat"
        # )

        # for condensate in condensate_names:
        #     ax[i, j].plot(
        #         cond[condensate],
        #         cond["P(bar)"],
        #         color=cond_colors[condensate],
        #         ls="dashed",
        #         zorder=-1,
        #         lw=3,
        #     )

        ax[i, j].loglog() 
        ax[i, j].invert_yaxis()
        ax[i, j].set_xlim([1e-14, 1e21])
        ax[i, j].set_ylim([1e2, 1e-3])
        
        ax[i, j].set_xlabel("Number density [cm⁻³]", fontsize=14)
        ax[i, j].set_ylabel("Pressure [bar]", fontsize=14)
        ax[i, j].set_title(f"{metallicity}x solar, {melt_frac}% melt")

# handles = [
#     Line2D([0], [0], ls="solid", color="k", label="Atmodeller", lw=1),
#     Line2D([0], [0], ls="dashed", color="k", label="FastChem", lw=3),
# ]
# ax[0, 0].legend(loc="lower left", handles=handles, framealpha=1)

labelLines(
    ax[0, 0].get_lines(),
    align=False,
    xvals=[1e17, 1e-11, 1e17, 1e17, 1e11],
)

labelLines(
    ax[0, 1].get_lines(),
    align=False,
    xvals=[1e18, 1e4, 1e16, 1e-9, 1e14],
)

labelLines(
    ax[1, 0].get_lines(),
    align=False,
    xvals=[1e16, 1e-11, 1e14, 1e18, 1e12],
)

labelLines(
    ax[1, 1].get_lines(),
    align=False,
    xvals=[1e17, 1e4, 1e-12, 1e-11, 1e-10],
)

fig.suptitle(
    r"Equilibrium Condensate Composition in the Atmosphere", fontsize=16
)

plt.savefig("HHeCNOSi_atmosphere_condensates.pdf", bbox_inches="tight")
plt.savefig("HHeCNOSi_atmosphere_condensates.png", bbox_inches="tight")
plt.show()

## D. SiH4/CH4 and Si/C Ratios at 10 mbar

### Retrieve Data

In [ ]:
# Define ranges
mantle_melt_fractions = [1, 0.3, 0.1, 0.03, 0.01, 0]
metallicities = [1, 3, 10, 30, 100]

# Dictionary to store results
results = {}

for melt_fraction in mantle_melt_fractions:
    for metallicity in metallicities:
        # Generate filename dynamically
        filename = f"HHeCNOSi_atm_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}_1wtH.xlsx"
        
        # Read data from Excel
        tot_pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
        C_moles = pd.read_excel(filename, sheet_name="element_C")["gas_number"]
        Si_moles = pd.read_excel(filename, sheet_name="element_Si")["gas_number"]
        SiH4_ratio = pd.read_excel(filename, sheet_name="H4Si_g")["volume_mixing_ratio"]
        CH4_ratio = pd.read_excel(filename, sheet_name="CH4_g")["volume_mixing_ratio"]
        
        # Create interpolation functions
        func_SiC = interp1d(tot_pressure, Si_moles / C_moles)
        func_SiH4CH4 = interp1d(tot_pressure, SiH4_ratio / CH4_ratio)
        
        # Store everything in a dictionary
        results[(melt_fraction, metallicity)] = {
            "pressure": tot_pressure,
            "func_SiC": func_SiC,
            "func_SiH4CH4": func_SiH4CH4
        }


### Plot Si/C

In [ ]:
mantle_melt_fractions = [1, 0.3, 0.1, 0.03, 0.01, 0]
metallicities = [1, 3, 10, 30, 100]

# Prepare meshgrid
x = [m * 100 for m in mantle_melt_fractions]  # convert to percent
X, Y = np.meshgrid(x, metallicities, indexing="ij")
xpoints, ypoints = X.flatten(), Y.flatten()

# Compute z-values 
pressure_10mbar = 1e-2  # 10 mbar
zpoints = [results[(melt, metal)]["func_SiC"](pressure_10mbar)
           for melt in mantle_melt_fractions for metal in metallicities]
zpoints = np.array(zpoints)

# Plot
fig, ax1 = plt.subplots(figsize=(6, 4.5), tight_layout=True)
sc = ax1.scatter(xpoints, ypoints, c=zpoints, cmap="viridis", norm=LogNorm(vmin=1e-17, vmax=1e-2))
ax1.set_xscale("linear")
ax1.set_yscale("log")
ax1.set_xlim(-0.2, 0.9)
ax1.spines["right"].set_visible(False)
ax1.yaxis.set_ticks_position("left")

ax1.set_xticks([0])
ax1.set_xticklabels(["0"])

ax1.set_ylabel(r"Accreted Gas Metallicity [$\times$ solar]", fontsize=14)

# Divider for shared y-axis
divider = make_axes_locatable(ax1)
axLog = divider.append_axes("right", size=3.3, pad=0.1, sharey=ax1)

axLog.scatter(xpoints, ypoints, c=zpoints, cmap="viridis", norm=LogNorm(vmin=1e-17, vmax=1e-2))
axLog.set_xscale("log")
axLog.set_yscale("log")
axLog.set_xlim(0.9, 200)
axLog.spines["left"].set_visible(False)
axLog.yaxis.set_ticks_position("right")
axLog.yaxis.set_visible(False)

axLog.set_xlabel(r"Mantle Melt [%]", fontsize=14)

# Colorbar
cbar_ax = divider.append_axes("right", size="20%", pad=0.1)
cbar = plt.colorbar(sc, cax=cbar_ax)
cbar.set_ticks([1e-17, 1e-12, 1e-7, 1e-2])
cbar.set_ticklabels(["<1e-17", "1e-12", "1e-7", ">1e-2"])
cbar.set_label("Si/C (molar ratio)", fontsize=14)

# Labels 
text_kwargs = dict(
    textcoords="offset points",
    xytext=(4, 0),            # 4-point horizontal offset
    ha="left", va="center",
    fontsize=8, color="black",
    clip_on=True,
)

# Label points 
for x0, y0, z0 in zip(xpoints, ypoints, zpoints):
    label = f"{z0:.0e}" #fmt_val(z0)
    if x0 == 0:  # the left linear-x panel
        ax1.annotate(label, (0, y0), **text_kwargs)
    else:        # the right log-x panel 
        axLog.annotate(label, (x0, y0), **text_kwargs)

fig.suptitle(f"(a) Si/C ratio at 10 mbar (TOI-421b)", fontsize=16)

# Save data
df = pd.DataFrame(zpoints.reshape(len(mantle_melt_fractions), len(metallicities)),
                  index=x, columns=metallicities)
df.to_excel(f"HHeCNOSi_SiCratios.xlsx")

plt.savefig(f"HHeCNOSi_SiCratios.pdf", bbox_inches="tight")
plt.savefig(f"HHeCNOSi_SiCratios.png", bbox_inches="tight")
plt.show()


### Plot SiH4/CH4

In [ ]:
mantle_melt_fractions = [1, 0.3, 0.1, 0.03, 0.01, 0]
metallicities = [1, 3, 10, 30, 100]

# Prepare meshgrid
x = [m * 100 for m in mantle_melt_fractions]  # convert to percent
X, Y = np.meshgrid(x, metallicities, indexing="ij")
xpoints, ypoints = X.flatten(), Y.flatten()

# Compute z-values 
pressure_10mbar = 1e-2  # 10 mbar
zpoints = [results[(melt, metal)]["func_SiH4CH4"](pressure_10mbar)
           for melt in mantle_melt_fractions for metal in metallicities]
zpoints = np.array(zpoints)

# Plot
fig, ax1 = plt.subplots(figsize=(6, 4.5), tight_layout=True)
sc = ax1.scatter(xpoints, ypoints, c=zpoints, cmap="viridis", norm=LogNorm(vmin=1e-17, vmax=1e-2))
ax1.set_xscale("linear")
ax1.set_yscale("log")
ax1.set_xlim(-0.2, 0.9)
ax1.spines["right"].set_visible(False)
ax1.yaxis.set_ticks_position("left")

ax1.set_xticks([0])
ax1.set_xticklabels(["0"])

ax1.set_ylabel(r"Accreted Gas Metallicity [$\times$ solar]", fontsize=14)

# Divider for shared y-axis
divider = make_axes_locatable(ax1)
axLog = divider.append_axes("right", size=3.3, pad=0.1, sharey=ax1)

axLog.scatter(xpoints, ypoints, c=zpoints, cmap="viridis", norm=LogNorm(vmin=1e-17, vmax=1e-2))
axLog.set_xscale("log")
axLog.set_yscale("log")
axLog.set_xlim(0.9, 200)
axLog.spines["left"].set_visible(False)
axLog.yaxis.set_ticks_position("right")
axLog.yaxis.set_visible(False)

axLog.set_xlabel(r"Mantle Melt [%]", fontsize=14)

# Colorbar
cbar_ax = divider.append_axes("right", size="20%", pad=0.1)
cbar = plt.colorbar(sc, cax=cbar_ax)
cbar.set_ticks([1e-17, 1e-12, 1e-7, 1e-2])
cbar.set_ticklabels(["<1e-17", "1e-12", "1e-7", ">1e-2"])
cbar.set_label(r"SiH$_4$/CH$_4$ (molar ratio)", fontsize=14)

# Labels 
text_kwargs = dict(
    textcoords="offset points",
    xytext=(4, 0),            # 4-point horizontal offset
    ha="left", va="center",
    fontsize=8, color="black",
    clip_on=True,
)

# Label points 
for x0, y0, z0 in zip(xpoints, ypoints, zpoints):
    label = f"{z0:.0e}" #fmt_val(z0)
    if x0 == 0:  # the left linear-x panel
        ax1.annotate(label, (0, y0), **text_kwargs)
    else:        # the right log-x panel 
        axLog.annotate(label, (x0, y0), **text_kwargs)

fig.suptitle(f"(b) SiH$_4$/CH$_4$ ratio at 10 mbar (TOI-421b)", fontsize=16)

# Save data
df = pd.DataFrame(zpoints.reshape(len(mantle_melt_fractions), len(metallicities)),
                  index=x, columns=metallicities)
df.to_excel(f"HHeCNOSi_SiH4Cratios.xlsx")

plt.savefig(f"HHeCNOSi_SiH4Cratios.pdf", bbox_inches="tight")
plt.savefig(f"HHeCNOSi_SiH4Cratios.png", bbox_inches="tight")
plt.show()


## E. Plot Si/C trends

### Plot Si/C trends

In [ ]:
# Define ranges
mantle_melt_fractions = [1, 0]
metallicities = [1, 100]
atmosphere_temperatures = [500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500]  # K
MEB_temperature = 3000 # K

# Dictionary to store results
results = {}

for melt_fraction in mantle_melt_fractions:
    for metallicity in metallicities:
        for atm_temp in atmosphere_temperatures:

            filename = f"HHeCNOSi_atm_magma_sol_real_{metallicity}xsolar_melt{round(melt_fraction * 100)}_1wtH_{atm_temp}K.xlsx"
        
            tot_pressure = pd.read_excel(filename, sheet_name="state")["pressure"]
            C_moles = pd.read_excel(filename, sheet_name="element_C")["gas_number"]
            Si_moles = pd.read_excel(filename, sheet_name="element_Si")["gas_number"]
            SiH4_moles = pd.read_excel(filename, sheet_name="H4Si_g")["gas_number"]
            SiH4_ratio = pd.read_excel(filename, sheet_name="H4Si_g")["volume_mixing_ratio"]
            CH4_ratio = pd.read_excel(filename, sheet_name="CH4_g")["volume_mixing_ratio"]
            
            func_SiC = interp1d(tot_pressure, Si_moles / C_moles)
            func_SiH4CH4 = interp1d(tot_pressure, SiH4_ratio / CH4_ratio)
            func_SiH4Si = interp1d(tot_pressure, SiH4_moles/ Si_moles)

            results[(melt_fraction, metallicity, atm_temp)] = {
                "pressure": tot_pressure,
                "func_SiC": func_SiC,
                "func_SiH4CH4": func_SiH4CH4,
                "func_SiH4Si": func_SiH4Si
            }


SiCratio_dict_1xsolar_melt100 = {}
SiCratio_dict_1xsolar_melt0 = {}
SiCratio_dict_100xsolar_melt100 = {}
SiCratio_dict_100xsolar_melt0 = {}
SiH4Siratio_dict_1xsolar_melt100 = {}
SiH4Siratio_dict_1xsolar_melt0 = {}
SiH4Siratio_dict_100xsolar_melt100 = {}
SiH4Siratio_dict_100xsolar_melt0 = {}

pressure_10mbar = 1e-2  # 10 mbar

for atm_temp in atmosphere_temperatures:

    filename = f"HHeCNOSi_SiCratios_{MEB_temperature}K_{atm_temp}K.xlsx"

    SiCratio_dict_1xsolar_melt100[atm_temp] = results[(1, 1, atm_temp)]["func_SiC"](pressure_10mbar)
    SiCratio_dict_1xsolar_melt0[atm_temp] = results[(0, 1, atm_temp)]["func_SiC"](pressure_10mbar)
    SiCratio_dict_100xsolar_melt100[atm_temp] = results[(1, 100, atm_temp)]["func_SiC"](pressure_10mbar)
    SiCratio_dict_100xsolar_melt0[atm_temp] = results[(0, 100, atm_temp)]["func_SiC"](pressure_10mbar)
    SiH4Siratio_dict_1xsolar_melt100[atm_temp] = results[(1, 1, atm_temp)]["func_SiH4Si"](pressure_10mbar)
    SiH4Siratio_dict_1xsolar_melt0[atm_temp] = results[(0, 1, atm_temp)]["func_SiH4Si"](pressure_10mbar)
    SiH4Siratio_dict_100xsolar_melt100[atm_temp] = results[(1, 100, atm_temp)]["func_SiH4Si"](pressure_10mbar)
    SiH4Siratio_dict_100xsolar_melt0[atm_temp] = results[(0, 100, atm_temp)]["func_SiH4Si"](pressure_10mbar)

funcRevSiC_1xsolar_melt100 = interp1d(atmosphere_temperatures, 
                                      np.array(list(SiCratio_dict_1xsolar_melt100.values())))
funcRevSiC_1xsolar_melt0 = interp1d(atmosphere_temperatures, 
                                    np.array(list(SiCratio_dict_1xsolar_melt0.values())))
funcRevSiC_100xsolar_melt100 = interp1d(atmosphere_temperatures, 
                                        np.array(list(SiCratio_dict_100xsolar_melt100.values())))
funcRevSiC_100xsolar_melt0 = interp1d(atmosphere_temperatures, 
                                      np.array(list(SiCratio_dict_100xsolar_melt0.values())))

funcSiH4Si_1xsolar_melt100 = interp1d(np.array(list(SiH4Siratio_dict_1xsolar_melt100.values())),
    atmosphere_temperatures)
funcSiH4Si_1xsolar_melt0 = interp1d(np.array(list(SiH4Siratio_dict_1xsolar_melt0.values())),
    atmosphere_temperatures)
funcSiH4Si_100xsolar_melt100 = interp1d(np.array(list(SiH4Siratio_dict_100xsolar_melt100.values())),
    atmosphere_temperatures)
funcSiH4Si_100xsolar_melt0 = interp1d(np.array(list(SiH4Siratio_dict_100xsolar_melt0.values())),
    atmosphere_temperatures)

fig, ax1 = plt.subplots(1, figsize=(6, 4.5), tight_layout="True")

ax1.plot(SiCratio_dict_1xsolar_melt100.keys(), SiCratio_dict_1xsolar_melt100.values(),
color="darkgreen", lw="3", ls='-', label="1x solar, 100% melt")

ax1.plot(SiCratio_dict_1xsolar_melt0.keys(), SiCratio_dict_1xsolar_melt0.values(),
color="darkgreen", lw="3", ls=':', label="1x solar, 0% melt")

ax1.plot(SiCratio_dict_100xsolar_melt100.keys(), SiCratio_dict_100xsolar_melt100.values(),
color="purple", lw="3", ls='-', label="100x solar, 100% melt")

ax1.plot(SiCratio_dict_100xsolar_melt0.keys(), SiCratio_dict_100xsolar_melt0.values(),
color="purple", lw="3", ls=':', label="100x solar, 0% melt")


refs = [
    (500, "TOI-1801b"),
    (550, "GJ 1214b"),
    (920,  "TOI-421b"),
    (1040, "TOI-125b"),
    (830,  "TOI-125c"),
    (640,  "TOI-125d"),
    (1250, "TOI-824b"),
    (870,  "TOI-451c"),
    (720,  "TOI-451d"),
    (1300, "HD 86226c"),
    (1450, "TOI-4010b"),
]

for xval, name in refs:

    ax1.axvline(xval, color="0.6", lw=1.2, ls="--", alpha=0.8, zorder=0)
    
    if name == "TOI-421b":
        ax1.annotate(
            name,
            xy=(xval, 0.8),              
            xycoords=("data", "axes fraction"),
            xytext=(3, 0), textcoords="offset points",
            rotation=90, rotation_mode="anchor",
            ha="left", va="bottom",
            fontsize=9, color="purple",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7) 
        )
    else:
        ax1.annotate(
            name,
            xy=(xval, 0.8),              
            xycoords=("data", "axes fraction"),
            xytext=(3, 0), textcoords="offset points",
            rotation=90, rotation_mode="anchor",
            ha="left", va="bottom",
            fontsize=9, color="0.25",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7) 
        )

ax1.set_xscale("linear")
ax1.set_xlim(450, 1550)
ax1.set_yscale("log")
ax1.set_ylim(1e-30, 1e3)
ax1.set_xlabel(r"Planet Equilibrium Temperature [K]", fontsize=14)
ax1.set_ylabel(r"Si/C in the gas phase (molar ratio)", fontsize=14)

ax1.fill([500, 700, funcSiH4Si_100xsolar_melt100(0.5), funcSiH4Si_1xsolar_melt100(0.5), 700, 500], 
          [SiCratio_dict_100xsolar_melt100[500], SiCratio_dict_100xsolar_melt100[700], 1e-10, 
           funcRevSiC_1xsolar_melt100(funcSiH4Si_100xsolar_melt100(0.5)), 
           SiCratio_dict_1xsolar_melt100[700], SiCratio_dict_1xsolar_melt100[500]], 
           color='pink', alpha=0.5, edgecolor='none', linewidth=1.5)

ax1.text(600, 1e-9, r"$X_{\mathrm{SiH_4}} > X_{\mathrm{SiO}}$", fontsize=14)

ax1.legend(fontsize=10, handlelength=3)

fig.suptitle(f"Si/C ratio at 10 mbar for sub-Neptunes", fontsize=14)

plt.savefig(f"HHeCNOSi_SiCratios_trends.pdf", bbox_inches="tight")
plt.savefig(f"HHeCNOSi_SiCratios_trends.png", bbox_inches="tight")
plt.show()